# SciBERT fine-tuning: ASEAN S&T field classifier

Fine-tunes `allenai/scibert_scivocab_uncased` on OpenAlex publication titles and abstracts to classify ASEAN S&T academic output into 21 fields (OECD Frascati Manual taxonomy).

**Input:** `training_dataset.csv` (OpenAlex publications labeled with ASJC field)  
**Output:** `scibert_classifier/` (fine-tuned model checkpoint)  
**Environment:** Google Colab, GPU runtime required (Runtime > Change runtime type > T4 GPU)

## Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Install dependencies

In [ ]:
%pip install transformers datasets scikit-learn pandas tqdm --quiet

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Load data

Mounts Google Drive and loads `training_dataset.csv`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/training_dataset.csv"

In [ ]:
df_full = pd.read_csv(CSV_PATH)
print(f"total: {len(df_full):,} rows | fields: {df_full['label'].nunique()}")
df_full['label'].value_counts()

## Sample for training

Caps each field at 5,000 records to balance classes and stay within Colab session limits.

In [ ]:
MAX_PER_FIELD = 5000

df = (
    df_full
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_PER_FIELD), random_state=42), include_groups=False)
    .reset_index(drop=True)
)
print(f"sampled total: {len(df):,}")
df['label'].value_counts()

## Label encoding

In [ ]:
labels = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(labels)

df['label_idx'] = df['label'].map(label2id)
print(f"classes: {NUM_LABELS}")

## Train / val / test split (80 / 10 / 10)

Stratified to preserve class distribution across all three partitions.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_idx']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label_idx']
)

print(f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")

## Tokenizer and dataset

In [ ]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class PaperDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

In [ ]:
train_dataset = PaperDataset(train_df['text'], train_df['label_idx'], tokenizer)
val_dataset = PaperDataset(val_df['text'], val_df['label_idx'], tokenizer)
test_dataset = PaperDataset(test_df['text'], test_df['label_idx'], tokenizer)

print(f"train: {len(train_dataset):,} | val: {len(val_dataset):,} | test: {len(test_dataset):,}")

## Load model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

## Training configuration

In [ ]:
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

print(f"batch size: {BATCH_SIZE} | epochs: {EPOCHS} | total steps: {total_steps:,}")

## Train

Approx. 2-3 hours on a T4 GPU.

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.extend(outputs.logits.argmax(-1).cpu().tolist())
            trues.extend(labels.cpu().tolist())
    acc = sum(p == t for p, t in zip(preds, trues)) / len(trues)
    return acc, preds, trues

In [ ]:
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val_acc, _, _ = evaluate(model, val_loader)
    print(f"epoch {epoch + 1} | loss: {avg_loss:.4f} | val acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained("scibert_classifier")
        tokenizer.save_pretrained("scibert_classifier")

print(f"best val accuracy: {best_val_acc:.4f}")

## Download checkpoint

Test-set evaluation (accuracy, per-field F1, confusion matrix) is in `2_scibert_evaluation.ipynb`.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("scibert_classifier", "zip", "scibert_classifier")
files.download("scibert_classifier.zip")